In [7]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

final_daily_df = pd.read_parquet(
    PROCESSED_DIR / "air_pollution_weather_daily_master.parquet"
)

final_daily_df["date"] = pd.to_datetime(final_daily_df["date"])

print(final_daily_df.shape)
display(final_daily_df[["station_id", "station_name", "date", "pm25"]].head())

(12271, 73)


,station_id,station_name,date,pm25
0,2919,Gazi Baba,2016-12-11,NaN
1,2919,Gazi Baba,2016-12-12,NaN
2,2919,Gazi Baba,2016-12-13,NaN
3,2919,Gazi Baba,2016-12-14,NaN
4,2919,Gazi Baba,2016-12-15,NaN


In [8]:
pm25_yearly_coverage_df = (
    final_daily_df
    .assign(
        year=final_daily_df["date"].dt.year,
        pm25_available=final_daily_df["pm25"].notna()
    )
    .groupby(
        ["station_id", "station_name", "year"],
        as_index=False
    )
    .agg(
        total_station_days=("date", "size"),
        pm25_days_available=("pm25_available", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

pm25_yearly_coverage_df["pm25_days_missing"] = (
    pm25_yearly_coverage_df["total_station_days"]
    - pm25_yearly_coverage_df["pm25_days_available"]
)

pm25_yearly_coverage_df["pm25_coverage_percent"] = (
    pm25_yearly_coverage_df["pm25_days_available"]
    / pm25_yearly_coverage_df["total_station_days"]
    * 100
).round(2)

pm25_yearly_coverage_df = (
    pm25_yearly_coverage_df
    .sort_values(["station_name", "year"])
    .reset_index(drop=True)
)

display(pm25_yearly_coverage_df)

,station_id,station_name,year,total_station_days,pm25_days_available,first_date,last_date,pm25_days_missing,pm25_coverage_percent
0,2996,Centar,2016,8,0,2016-12-11,2016-12-18,8,0.00
1,2996,Centar,2017,187,0,2017-02-19,2017-12-31,187,0.00
2,2996,Centar,2018,255,185,2018-01-01,2018-12-31,70,72.55
3,2996,Centar,2019,305,239,2019-01-02,2019-12-31,66,78.36
4,2996,Centar,2020,265,259,2020-01-01,2020-12-30,6,97.74
5,2996,Centar,2021,239,202,2021-01-01,2021-12-31,37,84.52
6,2996,Centar,2022,266,185,2022-01-01,2022-12-20,81,69.55
7,2996,Centar,2023,199,0,2023-01-04,2023-12-31,199,0.00
8,2996,Centar,2024,195,122,2024-01-01,2024-08-25,73,62.56
9,2996,Centar,2025,92,92,2025-09-19,2025-12-31,0,100.00


In [9]:
pm25_yearly_coverage_df = (
    final_daily_df
    .assign(
        year=final_daily_df["date"].dt.year,
        pm25_available=final_daily_df["pm25"].notna()
    )
    .groupby(
        ["station_id", "station_name", "year"],
        as_index=False
    )
    .agg(
        total_station_days=("date", "size"),
        pm25_days_available=("pm25_available", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

pm25_yearly_coverage_df["pm25_days_missing"] = (
    pm25_yearly_coverage_df["total_station_days"]
    - pm25_yearly_coverage_df["pm25_days_available"]
)

pm25_yearly_coverage_df["pm25_coverage_percent"] = (
    pm25_yearly_coverage_df["pm25_days_available"]
    / pm25_yearly_coverage_df["total_station_days"]
    * 100
).round(2)

pm25_yearly_coverage_df = (
    pm25_yearly_coverage_df
    .sort_values(["station_name", "year"])
    .reset_index(drop=True)
)

display(pm25_yearly_coverage_df)

,station_id,station_name,year,total_station_days,pm25_days_available,first_date,last_date,pm25_days_missing,pm25_coverage_percent
0,2996,Centar,2016,8,0,2016-12-11,2016-12-18,8,0.00
1,2996,Centar,2017,187,0,2017-02-19,2017-12-31,187,0.00
2,2996,Centar,2018,255,185,2018-01-01,2018-12-31,70,72.55
3,2996,Centar,2019,305,239,2019-01-02,2019-12-31,66,78.36
4,2996,Centar,2020,265,259,2020-01-01,2020-12-30,6,97.74
5,2996,Centar,2021,239,202,2021-01-01,2021-12-31,37,84.52
6,2996,Centar,2022,266,185,2022-01-01,2022-12-20,81,69.55
7,2996,Centar,2023,199,0,2023-01-04,2023-12-31,199,0.00
8,2996,Centar,2024,195,122,2024-01-01,2024-08-25,73,62.56
9,2996,Centar,2025,92,92,2025-09-19,2025-12-31,0,100.00


In [10]:
PROJECT_START = "2016-01-01"
PROJECT_END = "2025-12-31"

stations_df = (
    final_daily_df[
        ["station_id", "station_name"]
    ]
    .drop_duplicates()
    .sort_values("station_name")
    .reset_index(drop=True)
)

all_dates_df = pd.DataFrame({
    "date": pd.date_range(
        start=PROJECT_START,
        end=PROJECT_END,
        freq="D"
    )
})

station_calendar_df = (
    stations_df
    .merge(all_dates_df, how="cross")
)

pm25_values_df = (
    final_daily_df[
        [
            "station_id",
            "station_name",
            "date",
            "pm25",
            "pm25_valid_day",
            "pm25_observed_hours",
        ]
    ]
    .drop_duplicates(
        subset=[
            "station_id",
            "date",
        ]
    )
)

pm25_calendar_df = (
    station_calendar_df
    .merge(
        pm25_values_df,
        on=[
            "station_id",
            "station_name",
            "date",
        ],
        how="left",
    )
    .sort_values(["station_id", "date"])
    .reset_index(drop=True)
)

pm25_calendar_df["pm25_available"] = (
    pm25_calendar_df["pm25"].notna()
)

display(pm25_calendar_df.head())

,station_id,station_name,date,pm25,pm25_valid_day,pm25_observed_hours,pm25_available
0,2919,Gazi Baba,2016-01-01,NaN,NaN,NaN,False
1,2919,Gazi Baba,2016-01-02,NaN,NaN,NaN,False
2,2919,Gazi Baba,2016-01-03,NaN,NaN,NaN,False
3,2919,Gazi Baba,2016-01-04,NaN,NaN,NaN,False
4,2919,Gazi Baba,2016-01-05,NaN,NaN,NaN,False


In [11]:
def find_missing_periods(df, station_id, station_name):
    station_df = (
        df[
            df["station_id"] == station_id
        ]
        .sort_values("date")
        .copy()
    )

    station_df["is_missing"] = (
        ~station_df["pm25_available"]
    )

    station_df["missing_group"] = (
        station_df["is_missing"]
        .ne(station_df["is_missing"].shift())
        .cumsum()
    )

    periods = (
        station_df[
            station_df["is_missing"]
        ]
        .groupby("missing_group", as_index=False)
        .agg(
            missing_start=("date", "min"),
            missing_end=("date", "max"),
            missing_days=("date", "size"),
        )
    )

    periods["station_id"] = station_id
    periods["station_name"] = station_name

    return periods[
        [
            "station_id",
            "station_name",
            "missing_start",
            "missing_end",
            "missing_days",
        ]
    ]

missing_period_parts = []

for station in stations_df.itertuples(index=False):
    station_periods_df = find_missing_periods(
        pm25_calendar_df,
        station_id=station.station_id,
        station_name=station.station_name,
    )

    missing_period_parts.append(station_periods_df)

pm25_missing_periods_df = pd.concat(
    missing_period_parts,
    ignore_index=True,
)

pm25_missing_periods_df = (
    pm25_missing_periods_df
    .sort_values(
        ["station_name", "missing_start"]
    )
    .reset_index(drop=True)
)

display(pm25_missing_periods_df)

,station_id,station_name,missing_start,missing_end,missing_days
0,2996,Centar,2016-01-01,2018-03-06,796
1,2996,Centar,2018-03-10,2018-03-10,1
2,2996,Centar,2018-03-12,2018-04-18,38
3,2996,Centar,2018-05-04,2018-05-04,1
4,2996,Centar,2018-05-07,2018-05-07,1
...,...,...,...,...,...
465,2978,Miladinovci,2025-10-06,2025-10-07,2
466,2978,Miladinovci,2025-10-15,2025-10-16,2
467,2978,Miladinovci,2025-11-06,2025-11-07,2
468,2978,Miladinovci,2025-11-28,2025-12-01,4


In [12]:
LONG_GAP_DAYS = 7

pm25_long_gaps_df = pm25_missing_periods_df[
    pm25_missing_periods_df["missing_days"] >= LONG_GAP_DAYS
].copy()

pm25_long_gaps_df = (
    pm25_long_gaps_df
    .sort_values(
        ["station_name", "missing_days"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(pm25_long_gaps_df)

,station_id,station_name,missing_start,missing_end,missing_days
0,2996,Centar,2016-01-01,2018-03-06,796
1,2996,Centar,2022-09-24,2024-04-03,558
2,2996,Centar,2024-08-26,2025-09-18,389
3,2996,Centar,2019-07-01,2019-09-12,74
4,2996,Centar,2018-03-12,2018-04-18,38
5,2996,Centar,2021-04-01,2021-04-27,27
6,2996,Centar,2018-06-27,2018-07-11,15
7,2996,Centar,2018-12-02,2018-12-16,15
8,2996,Centar,2021-05-04,2021-05-16,13
9,2996,Centar,2020-07-05,2020-07-15,11
